<a href="https://colab.research.google.com/github/Phan-le-my-huyen/lung_project/blob/main/05_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "monai[nibabel,tqdm,einops]"

import os
BASE = '/content/drive/MyDrive/lung_project'
TAR  = f'{BASE}/data/Task06_Lung.tar'
ROOT = '/content/data/Task06_Lung'
if not os.path.exists(ROOT):
    !mkdir -p /content/data
    !tar -xf "{TAR}" -C /content/data --exclude='._*'
print('Sẵn sàng:', os.path.exists(ROOT))

Mounted at /content/drive
Sẵn sàng: True


In [4]:
import time, json, random, numpy as np, pandas as pd, torch
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, ScaleIntensityRanged, NormalizeIntensityd, GaussianSmoothd,
    SpatialPadd, RandCropByPosNegLabeld, RandFlipd, AsDiscrete)
from monai.data import PersistentDataset, Dataset, DataLoader
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

device = torch.device('cuda')
KEYS, PATCH = ['image', 'label'], (96, 96, 96)

folds = pd.read_csv(f'{BASE}/splits/folds_seed42.csv')
zs = pd.read_csv(f'{BASE}/preprocessing/zscore_stats_seed42.csv').set_index('fold')

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); set_determinism(seed=s)

def stage1(variant):                      # phần nặng, được đệm vào ổ tạm
    t = [LoadImaged(keys=KEYS), EnsureChannelFirstd(keys=KEYS),
         Orientationd(keys=KEYS, axcodes='RAS'),
         ScaleIntensityRanged(keys=['image'], a_min=-1000, a_max=400,
                              b_min=0.0, b_max=1.0, clip=True)]          # P0
    if variant != 'P0':
        t.append(Spacingd(keys=KEYS, pixdim=(1.0, 1.0, 1.0),
                          mode=('bilinear', 'nearest')))                 # P1
    t.append(EnsureTyped(keys=KEYS))
    return Compose(t)

def make_loader(cases, variant, train, mean=None, std=None):
    data = [{'image': f'{ROOT}/imagesTr/{c}.nii.gz',
             'label': f'{ROOT}/labelsTr/{c}.nii.gz'} for c in cases]
    cache = 'P0' if variant == 'P0' else 'P1'
    base = PersistentDataset(data, stage1(variant), cache_dir=f'/content/cache/{cache}')
    t = []
    if variant == 'P2':
        t.append(NormalizeIntensityd(keys=['image'], subtrahend=mean, divisor=std))
    if train:
        t += [SpatialPadd(keys=KEYS, spatial_size=PATCH),
              RandCropByPosNegLabeld(keys=KEYS, label_key='label', spatial_size=PATCH,
                                     pos=1, neg=1, num_samples=2),
              RandFlipd(keys=KEYS, prob=0.5, spatial_axis=0)]
    if variant == 'P3':
        t.append(GaussianSmoothd(keys=['image'], sigma=0.5))
    ds = Dataset(base, Compose(t))
    return DataLoader(ds, batch_size=1, shuffle=train, num_workers=2)
print('OK')

OK


In [5]:
post_pred, post_lab = AsDiscrete(argmax=True, to_onehot=2), AsDiscrete(to_onehot=2)
dice = DiceMetric(include_background=False, reduction='mean')

def evaluate(model, loader, cases):
    model.eval(); res = {}
    with torch.no_grad():
        for c, b in zip(cases, loader):
            x = b['image'].to(device)
            with torch.autocast('cuda'):
                out = sliding_window_inference(x, PATCH, 2, model, overlap=0.25,
                                               sw_device=device, device='cpu')
            dice(y_pred=post_pred(out[0])[None], y=post_lab(b['label'][0])[None])
            res[c] = dice.aggregate().item(); dice.reset()
    model.train()
    return res

def run(fold, variant, epochs=10, seed=42, lr=1e-4):
    seed_all(seed)
    tr = folds[folds.fold != fold]['case'].tolist()
    va = folds[folds.fold == fold]['case'].tolist()
    mean, std = float(zs.loc[fold, 'mean']), float(zs.loc[fold, 'std'])
    tl = make_loader(tr, variant, True, mean, std)
    vl = make_loader(va, variant, False, mean, std)

    model = UNet(spatial_dims=3, in_channels=1, out_channels=2,
                 channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
                 num_res_units=2, norm=Norm.INSTANCE).to(device)
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')

    log = {'fold': fold, 'variant': variant, 'seed': seed, 'loss': [], 'epoch_sec': []}
    for ep in range(epochs):
        t0, tot, n = time.time(), 0.0, 0
        for b in tl:
            x, y = b['image'].to(device), b['label'].to(device)
            opt.zero_grad()
            with torch.autocast('cuda'):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); n += 1
        log['loss'].append(tot / n); log['epoch_sec'].append(time.time() - t0)
        print(f'Epoch {ep+1}/{epochs}  loss={tot/n:.4f}  {time.time()-t0:.0f}s')

    t0 = time.time()
    per_case = evaluate(model, vl, va)
    log['val_sec'] = time.time() - t0
    log['dice_per_case'] = per_case
    log['dice_mean'] = float(np.mean(list(per_case.values())))
    print(f'Dice xác thực (fold {fold}, {variant}): {log["dice_mean"]:.4f}  ({log["val_sec"]:.0f}s)')
    try:
        json.dump(log, open(f'{BASE}/results/trial_{variant}_fold{fold}_seed{seed}.json', 'w'), indent=1)
    except Exception as e:
        print('Không lưu được lên Drive:', e)
    return log

In [6]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')

CUDA: True
2.14.1+cu130
Tesla T4


In [7]:
log = run(fold=0, variant='P1', epochs=10)

/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


RuntimeError: DataLoader worker (pid(s) 10425) exited unexpectedly